In [1]:
import os
from datetime import datetime

base_dir = "models/cross-encoder"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"run_{timestamp}"

output_dir = os.path.join(base_dir, run_name)

os.makedirs(output_dir, exist_ok=True)

print("Model will be saved to:", output_dir)

Model will be saved to: models/cross-encoder\run_20260305_210943


In [2]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('DiTy/cross-encoder-russian-msmarco')                 # Cross-Encoder (точный реранкинг)

In [3]:
import sentence_transformers
print(f"Текущая версия: {sentence_transformers.__version__}")

Текущая версия: 5.2.3


In [ ]:
from datasets import load_dataset
from sentence_transformers import InputExample

GOLDEN_DATSET_CSV = "data/golden/golden_pairs_without_negatives.csv"

# Загружаем датасет
dataset = load_dataset(
    "csv",
    data_files=GOLDEN_DATSET_CSV,
    encoding="utf-8-sig",
    sep='\t',
    engine="python"
)["train"]
# Оставляем только нужные колонки и переименовываем
dataset = dataset.rename_columns({
    "description": "text1",
    "post_text": "text2",
    "score": "label"
})

# Удаляем лишние колонки
dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in ["text1", "text2", "label"]]
)

# Приводим label к float (на всякий случай)
dataset = dataset.map(lambda x: {"label": float(x["label"])})
# dataset = dataset.select(range(500))


print(dataset[:10])

# Проверка диапазона (потому что лучше сейчас, чем потом)
assert all(0.0 <= x <= 1.0 for x in dataset["label"]), "Score должен быть в диапазоне 0-1"

# Преобразуем в InputExample
# train_samples = [
#     InputExample(texts=[row["text1"], row["text2"]], label=row["label"])
#     for row in dataset
# ]

{'text1': ['Уютный плед с вышивкой, выполненный из мягкой махры, подарит тепло и уют вашему дому. Элегантный узор ручной работы подчеркнет индивидуальность интерьера и добавит комфорта вашим вечерним посиделкам перед телевизором или книгой. Плед обладает приятной текстурой и приятен на ощупь, прекрасно сохранит тепло даже в прохладные вечера. Благодаря компактному размеру его удобно брать с собой в поездку или на дачу.', 'Настольная лампа с регулируемым углом наклона и мягким теплым светом идеально подойдет для чтения, выполнения домашних заданий или работы за компьютером. Прочный металлический каркас обеспечивает устойчивость и долговечность устройства, а поверхность лампы выполнена из экологичного материала, безопасного для здоровья. Подсветка регулируется плавно и легко, обеспечивая комфортное освещение рабочего места или зоны отдыха.', 'Мини-капсула ароматов «Парижские ночи» наполнена изысканными нотами ванили, лаванды и розы, создающими атмосферу элегантности и утонченности. Арома

In [5]:
print(len(dataset))

5071


In [6]:
# Train / Eval split (аналогично примеру из документации)
dataset_dict = dataset.train_test_split(test_size=0.2, seed=12)
train_dataset = dataset_dict["train"]
eval_dataset = dataset_dict["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['text1', 'text2', 'label'],
    num_rows: 4056
})
Dataset({
    features: ['text1', 'text2', 'label'],
    num_rows: 1015
})


In [7]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

loss = BinaryCrossEntropyLoss(model)

In [8]:
# Evaluator для регрессии (описание товара — пост — score 0..1)

from sentence_transformers.cross_encoder.evaluation import CrossEncoderCorrelationEvaluator

# Формируем список пар
sentence_pairs = [
    [row["text1"], row["text2"]]
    for row in eval_dataset
]

# Список gold-оценок
scores = [
    float(row["label"])
    for row in eval_dataset
]

evaluator = CrossEncoderCorrelationEvaluator(
    sentence_pairs=sentence_pairs,
    scores=scores,
    name="eval_correlation",
    batch_size=32,
)

results = evaluator(model)

primary_metric = evaluator.primary_metric
primary_score = results[evaluator.primary_metric]

print("Primary metric:", primary_metric)
print("Primary score:", primary_score)

Primary metric: eval_correlation_spearman
Primary score: 0.3847509248561655


In [9]:
from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments

args = CrossEncoderTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=False,
    bf16=False,
    load_best_model_at_end=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=10,
    run_name=run_name,
    seed=12,
)

In [10]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=evaluator,
)

trainer.train()

Step,Training Loss,Validation Loss,Correlation Pearson,Correlation Spearman
500,0.463800,0.582845,0.750716,0.727975


TrainOutput(global_step=508, training_loss=0.553964800722017, metrics={'train_runtime': 9692.959, 'train_samples_per_second': 1.674, 'train_steps_per_second': 0.052, 'total_flos': 0.0, 'train_loss': 0.553964800722017, 'epoch': 4.0})

In [11]:
final_output_dir = os.path.join(output_dir, "final")
model.save_pretrained(final_output_dir)

print("Final model saved to:", final_output_dir)

Final model saved to: models/cross-encoder\run_20260305_210943\final


In [12]:
# Загрузка обученной модели
from sentence_transformers import CrossEncoder

trained_model = CrossEncoder(final_output_dir)
print(f"Обученная модель загружена из: {final_output_dir}")

The tokenizer you are loading from 'models/cross-encoder\run_20260305_210943\final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Обученная модель загружена из: models/cross-encoder\run_20260305_210943\final


In [13]:
# Прогоняем evaluator на обученной модели
results_after = evaluator(trained_model)
primary_score_after = results_after[evaluator.primary_metric]

print("Primary metric:", evaluator.primary_metric)
print("Primary score ПОСЛЕ обучения:", primary_score_after)

Primary metric: eval_correlation_spearman
Primary score ПОСЛЕ обучения: 0.7279750532498214


In [14]:
# Сравнение метрик
print("=== СРАВНЕНИЕ PRIMARY METRIC (Spearman correlation) ===")
print(f"До обучения (pretrained):  {primary_score:.6f}")
print(f"После обучения (fine-tuned): {primary_score_after:.6f}")

delta = primary_score_after - primary_score
print(f"Изменение: {delta:+.6f} → {'УЛУЧШЕНИЕ' if delta > 0 else 'УХУДШЕНИЕ'}")

print("\nИнтерпретация:")
print("• Чем ближе к +1 — тем лучше ранжирование (описание товара ↔ пост)")
print("• Отрицательное значение = модель хуже случайного ранжирования")

=== СРАВНЕНИЕ PRIMARY METRIC (Spearman correlation) ===
До обучения (pretrained):  0.384751
После обучения (fine-tuned): 0.727975
Изменение: +0.343224 → УЛУЧШЕНИЕ

Интерпретация:
• Чем ближе к +1 — тем лучше ранжирование (описание товара ↔ пост)
• Отрицательное значение = модель хуже случайного ранжирования
